# Feature lịch sử seller không nhìn trộm tương lai

"Seller này hay bị review xấu" là một feature rất mạnh cho bài toán review xấu. Cách tính dễ nhất,
`groupby("seller_id").bad_review.mean()` trên toàn bộ bảng, dùng cả review của **tương lai** và
review của **chính đơn đang dự đoán**. Bài này tính lại feature đó đúng cách: tại lúc đơn vừa giao,
chỉ những review đã được trả lời trước lúc đó mới được dùng.

**Cách làm bài**
- `seller_history` được chấm bằng test. Các cell `explore` cho bạn số liệu để viết `REPORT.md`.
- Chấm: `uv run pytest tracks/data-science/modules/10-feature-engineering/exercises/advanced/01-point-in-time-seller -q`

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, TargetEncoder

In [ ]:
# Cell viết sẵn: đọc 8 bảng Olist, tạo một dòng cho mỗi đơn giao xong có review.
# Thời điểm dự đoán: lúc đơn vừa giao xong (order_delivered_customer_date), trước khi khách review.
DATES = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
         "order_delivered_customer_date", "order_estimated_delivery_date"]
ZIP = {"customer_zip_code_prefix": str, "seller_zip_code_prefix": str, "geolocation_zip_code_prefix": str}


def _haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = (np.radians(v) for v in (lat1, lon1, lat2, lon2))
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))


def load_olist_table(raw) -> pd.DataFrame:
    raw = Path(raw)
    orders = pd.read_csv(raw / "olist_orders_dataset.csv", parse_dates=DATES)
    reviews = pd.read_csv(raw / "olist_order_reviews_dataset.csv",
                          parse_dates=["review_creation_date", "review_answer_timestamp"])
    customers = pd.read_csv(raw / "olist_customers_dataset.csv", dtype=ZIP)
    items = pd.read_csv(raw / "olist_order_items_dataset.csv")
    products = pd.read_csv(raw / "olist_products_dataset.csv")
    sellers = pd.read_csv(raw / "olist_sellers_dataset.csv", dtype=ZIP)
    payments = pd.read_csv(raw / "olist_order_payments_dataset.csv")
    geo = pd.read_csv(raw / "olist_geolocation_dataset.csv", dtype=ZIP)

    delivered = orders[(orders.order_status == "delivered") & orders.order_delivered_customer_date.notna()]
    latest = reviews.sort_values("review_answer_timestamp").drop_duplicates("order_id", keep="last")
    df = delivered.merge(latest, on="order_id").merge(customers, on="customer_id")
    df["bad_review"] = (df.review_score <= 2).astype(int)

    day = pd.Timedelta(days=1)
    df["purchase_hour"] = df.order_purchase_timestamp.dt.hour
    df["purchase_dow"] = df.order_purchase_timestamp.dt.day_of_week
    df["promised_days"] = (df.order_estimated_delivery_date - df.order_purchase_timestamp) / day
    df["delivery_days"] = (df.order_delivered_customer_date - df.order_purchase_timestamp) / day
    df["delay_days"] = (df.order_delivered_customer_date - df.order_estimated_delivery_date) / day
    df["approve_hours"] = (df.order_approved_at - df.order_purchase_timestamp) / pd.Timedelta(hours=1)

    items = items.merge(products, on="product_id", how="left").merge(sellers, on="seller_id", how="left")
    per_order = items.groupby("order_id").agg(
        n_items=("order_item_id", "size"), n_sellers=("seller_id", "nunique"),
        price=("price", "sum"), freight=("freight_value", "sum"), weight_g=("product_weight_g", "sum"),
        photos=("product_photos_qty", "mean"), desc_len=("product_description_lenght", "mean"))
    first = items[items.order_item_id == 1][
        ["order_id", "seller_id", "product_category_name", "seller_state", "seller_zip_code_prefix"]]
    df = df.merge(per_order, on="order_id").merge(first, on="order_id")
    df["freight_ratio"] = df.freight / (df.price + df.freight)
    main_payment = payments.sort_values("payment_value", ascending=False).drop_duplicates("order_id")
    df = df.merge(main_payment[["order_id", "payment_type", "payment_installments"]], on="order_id", how="left")

    ok = geo.geolocation_lat.between(-34, 6) & geo.geolocation_lng.between(-74, -34)
    centroid = geo[ok].groupby("geolocation_zip_code_prefix")[["geolocation_lat", "geolocation_lng"]].median()
    c = centroid.reindex(df.customer_zip_code_prefix).to_numpy()
    s = centroid.reindex(df.seller_zip_code_prefix).to_numpy()
    df["distance_km"] = _haversine_km(c[:, 0], c[:, 1], s[:, 0], s[:, 1])
    df["same_state"] = (df.customer_state == df.seller_state).astype(int)
    return df.sort_values("order_purchase_timestamp", kind="stable").reset_index(drop=True)


def time_split(df: pd.DataFrame) -> dict:
    t = df.order_purchase_timestamp
    return {"train": df[t < "2018-01-01"],
            "val": df[(t >= "2018-01-01") & (t < "2018-04-01")],
            "test": df[(t >= "2018-04-01") & (t < "2018-09-01")]}

In [ ]:
# explore: tìm gốc repo và đọc bảng feature Olist (tải bằng: uv run python data/download_olist.py)
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
table = load_olist_table(ROOT / "data" / "raw")
splits = time_split(table)
train, val, test = splits["train"], splits["val"], splits["test"]
print(table.shape, {k: len(v) for k, v in splits.items()})

## 1. Hàm point-in-time

Cột đầu vào được đổi tên cho tổng quát: `prediction_time` là lúc dự đoán (với Olist:
`order_delivered_customer_date`), `label_time` là lúc nhãn được biết (`review_answer_timestamp`).
Chú ý ba bẫy: so sánh **chặt**; review của chính đơn đó (khoảng 5% đơn Olist có review trả lời
trước cả ngày giao); và tốc độ (seller lớn nhất có hơn một nghìn đơn).

In [ ]:
def seller_history(df: pd.DataFrame, prior: float, smoothing: float = 20.0) -> pd.DataFrame:
    """Lịch sử review của seller tính tại thời điểm dự đoán của từng dòng (point-in-time).

    df có cột seller_id, prediction_time, label_time (có thể NaT), bad_review; thứ tự dòng bất kỳ.
    Với mỗi dòng i, "quá khứ" là các dòng j khác i, cùng seller, có label_time_j < prediction_time_i.
    Trả DataFrame cùng index với df:
      seller_past_reviews   số dòng quá khứ (int)
      seller_past_bad_rate  (số review xấu quá khứ + smoothing · prior) / (seller_past_reviews + smoothing)
    Phải chạy nhanh: 100 nghìn dòng dưới vài giây.
    """
    # TODO
    raise NotImplementedError

In [ ]:
# explore: gắn feature vào bảng Olist
prior = train.bad_review.mean()
hist = seller_history(table.rename(columns={"order_delivered_customer_date": "prediction_time",
                                            "review_answer_timestamp": "label_time"}), prior)
table = table.join(hist)
splits = time_split(table)
train, val, test = splits["train"], splits["val"], splits["test"]
print(hist.describe().round(3))
print("val: đơn của seller chưa có lịch sử:", round((val.seller_past_reviews == 0).mean(), 4))

## 2. So bốn cách đưa thông tin seller vào model

Cùng model logistic của bài Intermediate 02, chỉ khác feature về seller. Ghi bảng kết quả vào
`REPORT.md` và giải thích vì sao thứ tự trên validation và trên test lại như vậy.

In [ ]:
# explore: bốn biến thể, AUC val (fit trên train) và AUC test (fit lại trên train + val)
NUMERIC = ["purchase_hour", "purchase_dow", "promised_days", "delivery_days", "delay_days", "approve_hours",
           "n_items", "n_sellers", "price", "freight", "freight_ratio", "weight_g", "photos", "desc_len",
           "payment_installments", "distance_km", "same_state"]
CATEGORICAL = ["customer_state", "seller_state", "payment_type", "product_category_name"]


def model(numeric, high_card=()):
    parts = [("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), numeric),
             ("cat", make_pipeline(SimpleImputer(strategy="constant", fill_value="missing"),
                                   OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=20)), CATEGORICAL)]
    if high_card:
        parts.append(("te", TargetEncoder(cv=KFold(5, shuffle=True, random_state=0)), list(high_card)))
    return make_pipeline(ColumnTransformer(parts), LogisticRegression(max_iter=2000))


naive_rate = table.groupby("seller_id").bad_review.mean()        # SAI: toàn bộ dữ liệu, cả tương lai
table["seller_rate_all"] = table.seller_id.map(naive_rate)
table["log_past_reviews"] = np.log1p(table.seller_past_reviews)
splits = time_split(table)
train, val, test = splits["train"], splits["val"], splits["test"]
dev = pd.concat([train, val])
variants = {
    "không có seller": model(NUMERIC),
    "target encoding seller_id": model(NUMERIC, ["seller_id"]),
    "tỉ lệ seller trên toàn bộ dữ liệu": model(NUMERIC + ["seller_rate_all"]),
    "lịch sử seller point-in-time": model(NUMERIC + ["seller_past_bad_rate", "log_past_reviews"]),
}
rows = {}
for name, pipe in variants.items():
    pipe.fit(train, train.bad_review)
    auc_val = roc_auc_score(val.bad_review, pipe.predict_proba(val)[:, 1])
    pipe.fit(dev, dev.bad_review)
    rows[name] = {"auc_val": auc_val, "auc_test": roc_auc_score(test.bad_review, pipe.predict_proba(test)[:, 1])}
pd.DataFrame(rows).T.round(4)

## 3. Viết `REPORT.md`

Theo khung trong `REPORT.md`. Người đọc là một DS khác trong team sẽ đưa feature này vào
production.